# GNN tuned-GFP runs (four operators, configurations 4 and 5)

Runs on a Kaggle GPU notebook: the four operators on configurations 4 and 5, the fixed hyperparameters (`conf/hyperparameters/gnn/fixed.yaml`), with the tuned GFP sheet (64 columns) in place of the default one (61 columns). Eight runs, about 8 hours.
- Settings (right panel): Accelerator = GPU, Internet = On.
- Input: the datasets `sandrokhizanishvili/hi-small-gnn` and `sandrokhizanishvili/hi-small-gfp-tuned`.
- Run All (or Save Version -> Save & Run All to run headless, 12 h limit).
- Each run writes `results.json`, `history.csv`, `curves.png`, `predictions.csv`, `best.pt`, `config.yaml` to `/kaggle/working/outputs/train/<model>/<run>_gfp-tuned/seed-42/`.

In [1]:
MODELS = 'gin,pna,gat,transformer'
FEATURE_CONFIGS = 'c4,c5'
SEED = 42

In [2]:
# The pinned torch / PyG stack known to work on Kaggle GPU; do not change versions.
!pip uninstall -y torch torchvision torchaudio torch-scatter torch-sparse pyg_lib
!pip install torch==2.8.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.8.0+cu126.html
!pip install torch_geometric lightgbm==4.6.0 hydra-core==1.3.2

Found existing installation: torch 2.11.0+cu130
Uninstalling torch-2.11.0+cu130:
  Successfully uninstalled torch-2.11.0+cu130
Found existing installation: torchvision 0.26.0+cu130
Uninstalling torchvision-0.26.0+cu130:
  Successfully uninstalled torchvision-0.26.0+cu130
Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130
Looking in indexes: https://download.pytorch.org/whl/cu126
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 358.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 390.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 453.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 412.5 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 363.7 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 366.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [3]:
# The code: clone once, pull on a rerun.
import os
if os.path.isdir('/kaggle/working/gnn_for_aml'):
    !cd /kaggle/working/gnn_for_aml && git pull
else:
    !git clone https://github.com/sandrokhizanishvili/gnn_for_aml.git /kaggle/working/gnn_for_aml
%cd /kaggle/working/gnn_for_aml

Cloning into '/kaggle/working/gnn_for_aml'...
remote: Enumerating objects: 181, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 181 (delta 76), reused 152 (delta 49), pack-reused 0 (from 0)
Receiving objects: 100% (181/181), 338.43 KiB | 2.27 MiB/s, done.
Resolving deltas: 100% (76/76), done.
/kaggle/working/gnn_for_aml


In [4]:
# One data folder: the graphs and gfp_variants/ linked from the two datasets (in /tmp, so the links stay out of the Output tab).
INPUT = '/kaggle/input/datasets/sandrokhizanishvili'
!mkdir -p /tmp/data
!ln -sf {INPUT}/hi-small-gnn/train_graph.pt {INPUT}/hi-small-gnn/val_graph.pt {INPUT}/hi-small-gnn/test_graph.pt {INPUT}/hi-small-gnn/feature_meta.json /tmp/data/
!ln -sfn {INPUT}/hi-small-gfp-tuned /tmp/data/gfp_variants
!ls -lL /tmp/data /tmp/data/gfp_variants

/tmp/data:
total 4284450
-rw-rw-rw- 1 root root       4717 Nov 10 20:28 feature_meta.json
drwxrwxrwx 1 root root          0 Nov 10 20:29 gfp_variants
-rw-rw-rw- 1 root root 1824938361 Nov 10 20:29 test_graph.pt
-rw-rw-rw- 1 root root 1099908869 Nov 10 20:29 train_graph.pt
-rw-rw-rw- 1 root root 1462423405 Nov 10 20:29 val_graph.pt

/tmp/data/gfp_variants:
total 1269311
-rw-rw-rw- 1 root root       1475 Nov 10 20:28 tuned_cols.json
-rw-rw-rw- 1 root root 1299772800 Nov 10 20:29 tuned.npy


In [5]:
# Environment test: versions, the GPU, one forward / backward pass per operator.
!python run.py mode=check machine=kaggle

python 3.13.16 | torch 2.8.0+cu126 | CUDA build 12.6 | device cuda
  torch_geometric 2.8.1
  torch_scatter 2.1.2+pt28cu126
  torch_sparse 0.6.18+pt28cu126
  lightgbm 4.6.0
  hydra-core 1.3.2
  hydra-submitit-launcher not installed (only needed on Cineca)
  numpy 2.1.3
  scipy 1.16.3
  pandas 2.3.3
  scikit-learn 1.6.1
  matplotlib 3.10.0
  psutil 5.9.5
/usr/local/lib/python3.13/dist-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
  gin         params   54,083 | loss 0.6760
  pna         params  176,833 | loss 0.7382
  gat         params   53,953 | loss 0.7751
  transformer params   70,337 | loss 0.7255
OK


In [6]:
# The eight runs, one after another. A finished run is skipped, so the cell can be rerun after an interruption.
!python run.py -m model={MODELS} feature_config={FEATURE_CONFIGS} gfp_sheet=tuned seed={SEED} machine=kaggle paths.data=/tmp/data

[2026-11-10 20:33:14,118][HYDRA] Launching 8 jobs locally
[2026-11-10 20:33:14,118][HYDRA] 	#0 : model=gin feature_config=c4 gfp_sheet=tuned seed=42 machine=kaggle paths.data=/tmp/data
=== train gin_mp-full_readout-full_dir-in_gfp-tuned seed 42 -> /kaggle/working/outputs/train/gin/gin_mp-full_readout-full_dir-in_gfp-tuned/seed-42
mode: train
seed: 42
trial: null
node_encoding: none
gfp_sheet: tuned
hyperparameters:
  for: gnn
  hidden: 128
  num_layers: 2
  dropout: 0.3
  norm: none
  residual: identity
  ffn: false
  num_neighbors:
  - 100
  - 100
  batch_size: 8192
  epochs: 20
  lr: 0.001
  weight_decay: 1.0e-05
  pos_weight: 8.0
  self_loops: true
  readout_layers: 2
  readout_width: 1.0
  lr_schedule: cosine
  num_workers: 0
  select_metric: pr_auc
model:
  name: gin
  kind: gnn
feature_config:
  id: 4
  mp: full
  readout: full
machine: kaggle
paths:
  data: /tmp/data
  outputs: /kaggle/working/outputs

edge_attr width 84 (base 20 + gfp 64, sheet tuned)
train nodes=515,070 edges=

In [7]:
# The metrics of every finished run, one row each.
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob('/kaggle/working/outputs/train/*/*_gfp-tuned/seed-*/results.json'))]
cols = ['run', 'best_epoch', 'threshold', 'params', 'invariant_params'] + [f'{s}_{m}' for s in ('train', 'val', 'test') for m in ('f1', 'precision', 'recall', 'pr_auc')]
pd.DataFrame(rows)[cols].round(4) if rows else "no finished run yet"

,run,best_epoch,threshold,params,invariant_params,train_f1,train_precision,train_recall,train_pr_auc,val_f1,val_precision,val_recall,val_pr_auc,test_f1,test_precision,test_recall,test_pr_auc
0,gat_mp-base_readout-full_dir-in_gfp-tuned,16,0.4849,116353,67585,0.4616,0.5089,0.4223,0.4410,0.5982,0.7894,0.4815,0.5367,0.5193,0.6713,0.4234,0.4808
1,gat_mp-full_readout-full_dir-in_gfp-tuned,11,0.4680,132737,67585,0.4641,0.5010,0.4323,0.4405,0.5853,0.8091,0.4584,0.5341,0.5116,0.6700,0.4138,0.4746
2,gin_mp-base_readout-full_dir-in_gfp-tuned,13,0.6306,116611,67587,0.5215,0.5670,0.4828,0.5138,0.6117,0.7617,0.5111,0.5562,0.5201,0.6694,0.4252,0.4816
3,gin_mp-full_readout-full_dir-in_gfp-tuned,12,0.5990,132995,67587,0.5741,0.7275,0.4741,0.5507,0.5984,0.8584,0.4593,0.5278,0.4808,0.7365,0.3570,0.4204
4,pna_mp-base_readout-full_dir-in_gfp-tuned,18,0.6992,607873,427265,0.6018,0.8443,0.4676,0.5778,0.6486,0.8584,0.5213,0.6075,0.6176,0.8108,0.4987,0.5703
5,pna_mp-full_readout-full_dir-in_gfp-tuned,13,0.6635,624257,427265,0.5860,0.7871,0.4667,0.5598,0.6525,0.8571,0.5268,0.5997,0.6122,0.8387,0.4821,0.5657
6,transformer_mp-base_readout-full_dir-in_gfp-tuned,15,0.6654,181889,133121,0.5802,0.7953,0.4567,0.5512,0.6351,0.8621,0.5028,0.5899,0.6114,0.8407,0.4803,0.5735
7,transformer_mp-full_readout-full_dir-in_gfp-tuned,14,0.6514,198273,133121,0.5933,0.7762,0.4802,0.5841,0.6421,0.8530,0.5148,0.5916,0.6210,0.8204,0.4996,0.5811


In [8]:
# A small archive for download: everything except the checkpoints and predictions (the Output tab has the full folder).
!cd /kaggle/working && rm -f results_small.zip && zip -qr results_small.zip outputs -x "*.pt" "*predictions.csv" && ls -lh results_small.zip

-rw-r--r-- 1 root root 1.6M Nov 11 04:07 results_small.zip


## After the run
- Output tab: download `results_small.zip` (metrics, curves, histories) or the whole `outputs` folder (adds `predictions.csv` and `best.pt`).
- Unzip into `gnn_for_aml/outputs/` on the laptop; commit the small files (`predictions.csv` and `*.pt` are gitignored); mark the row in `EXPERIMENTS.md`.